# Reporter titration plots

Per-reporter titration curves of mAP (distinctiveness and EBI) vs cells per guide across imaging modalities: live-cell fluorescence reporters, 4i, cp, and Phase. Each metric is shown on a full log scale and a linear close-up (100–1000 cells per guide). Combined-reporter aggregates are plotted separately in `combined_reporter_titration.ipynb`.

Input data is the merged per-reporter titration CSV produced by `data_preprocessing/figure_4.py`.

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["svg.fonttype"] = "none"

FIGURES_DIR = Path("../../output/figure_4")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## CSV paths

Merged per-reporter titration CSV (full log range concatenated with the 50–1000 cells/guide zoom range) curated into `../../data/figures/figure_4/` (see README).

In [ ]:
MERGED_CSV = Path("../../data/figures/figure_4/titration_individual_reporters.csv")

## Load and preprocess

Load the merged per-reporter titration curves, drop excluded 4i signals, and tag each row with its imaging modality (`live-cell` / `4i` / `cp`).

In [ ]:
EXCLUDE_SIGNALS = {"gH2AX (4i)", "NFkB (4i)", "RSP6 (4i)", "Rb (4i)"}

def classify_signal(name):
    if "(4i)" in name:
        return "4i"
    if "(cp)" in name:
        return "cp"
    return "live-cell"

df = pd.read_csv(MERGED_CSV)
df = df[~df["signal"].isin(EXCLUDE_SIGNALS)].reset_index(drop=True)
df["modality"] = df["signal"].map(classify_signal)

MODALITY_ORDER = ["4i", "cp", "live-cell"]
for modality in MODALITY_ORDER:
    n = df[df["modality"] == modality]["signal"].nunique()
    print(f"{modality:>10}: {n} channels")

## Shared plot styling

Per-modality color / alpha / linewidth used across every plot below.

In [ ]:
COLORS = {"4i": "#f28c3a", "cp": "#7b5ea7", "live-cell": "#999999", "livecell": "#555555"}
ALPHA  = {"4i": 1,         "cp": 1,         "live-cell": 0.5,       "livecell": 0.8}
LW     = {"4i": 1,         "cp": 1,         "live-cell": 1,         "livecell": 1}

## Plot 1 — Distinctiveness, full log scale

Per-reporter distinctiveness curves across the full cells-per-guide range, log-scaled. Phase is highlighted in black. The close-up inset region is shown in the next plot.

In [ ]:
PLOT1_LABELS = {"live-cell": "live-cell fluorescence", "4i": "4i", "cp": "cp", "Phase": "Phase"}

def titration_ax(ax, y_col, ylabel, title):
    phase_grp = None
    for modality in ["live-cell", "4i", "cp"]:
        first = True
        for signal, grp in df[df["modality"] == modality].groupby("signal"):
            if signal == "Phase":
                phase_grp = grp
                continue
            grp_s = grp.sort_values("cells_per_perturbation")
            ax.plot(
                grp_s["cells_per_guide"],
                grp_s[y_col],
                color=COLORS["live-cell"],
                alpha=ALPHA[modality],
                linewidth=LW[modality],
                marker="o",
                markersize=2,
                label=PLOT1_LABELS[modality] if first else "_nolegend_",
            )
            first = False
    if phase_grp is not None:
        grp_s = phase_grp.sort_values("cells_per_perturbation")
        ax.plot(
            grp_s["cells_per_guide"],
            grp_s[y_col],
            color="darkred",
            alpha=1.0,
            linewidth=2.0,
            marker="o",
            markersize=2,
            label=PLOT1_LABELS["Phase"],
            zorder=5,
        )
    ax.set_xscale("log")
    ax.set_xlabel("Cells per guide (log scale)")
    ax.set_xlim(1, 15_000)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    #ax.legend(frameon=False, fontsize=9)
    #ax.grid(alpha=0.3)

fig, ax = plt.subplots(figsize=(6, 5))
titration_ax(ax, "distinctiveness_map_mean", "Distinctiveness mAP", "Distinctiveness")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "titration_distinctiveness_full_scale.svg", bbox_inches="tight")
plt.show()

## Plot 2 — Distinctiveness, close-up inset (100–1000 cells/guide)

Linear close-up of plot 1, restricted to 100–1000 cells per guide via `xlim`. The denser sampling in this range comes from the zoom rows of the merged CSV.

In [ ]:
def titration_zoom_ax(ax, y_col, ylim, yticks):
    phase_grp = None
    for modality in ["live-cell", "4i", "cp"]:
        first = True
        for signal, grp in df[df["modality"] == modality].groupby("signal"):
            if signal == "Phase":
                phase_grp = grp
                continue
            grp_s = grp.sort_values("cells_per_perturbation")
            ax.plot(
                grp_s["cells_per_guide"],
                grp_s[y_col],
                color=COLORS["live-cell"],
                alpha=ALPHA[modality],
                linewidth=LW[modality],
                marker="o",
                markersize=3,
                label=PLOT1_LABELS[modality] if first else "_nolegend_",
            )
            first = False
    if phase_grp is not None:
        grp_s = phase_grp.sort_values("cells_per_perturbation")
        ax.plot(
            grp_s["cells_per_guide"],
            grp_s[y_col],
            color="black",
            alpha=1.0,
            linewidth=2.0,
            marker="o",
            markersize=4,
            label=PLOT1_LABELS["Phase"],
            zorder=5,
        )
    # ax.set_xscale("log")
    ax.set_xlim(100, 1_000)
    ax.set_xticks([100, 500, 1000], labels=["100", "500", "1000"])
    ax.set_xticks([], minor=True)
    ax.set_ylim(*ylim)
    ax.set_yticks(yticks)
    # Move x-axis to the top, ticks and labels facing outward
    ax.xaxis.set_ticks_position("top")
    ax.xaxis.set_label_position("top")
    ax.tick_params(
        axis="x", top=True, bottom=False, labeltop=True, labelbottom=False, direction="out"
    )

fig, ax = plt.subplots(figsize=(4, 3))
titration_zoom_ax(ax, "distinctiveness_map_mean", (0, 0.3), np.arange(0, 0.31, 0.1))
fig.tight_layout()
fig.savefig(FIGURES_DIR / "titration_distinctiveness.svg", bbox_inches="tight")
plt.show()

## Plot 3 — EBI, full log scale

Per-reporter EBI curves across the full cells-per-guide range. Phase highlighted in black; close-up inset shown in the next plot.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
titration_ax(ax, "ebi_map_mean", "EBI mAP", "EBI")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "titration_ebi_full_scale.svg", bbox_inches="tight")
plt.show()

## Plot 4 — EBI, close-up inset (100–1000 cells/guide)

Linear close-up of plot 3.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
titration_zoom_ax(ax, "ebi_map_mean", (0, 0.47), np.arange(0, 0.46, 0.1))
fig.tight_layout()
fig.savefig(FIGURES_DIR / "titration_ebi.svg", bbox_inches="tight")
plt.show()